# UC-11 — silver (Databricks)

Leest de zeven `bronze.*` UC-tabellen en schrijft `silver.*` Delta-tabellen met
dezelfde kolommen die `int_klantreis_events` verwacht. Deduplicatie + kolom-rename
(source_ts → ingestion_source_ts voor persona).

Spiegel van uc11_silver Fabric notebook, maar tegen UC i.p.v. OneLake paths.

In [ ]:
dbutils.widgets.text("catalog", "uwv_databricks")
catalog = dbutils.widgets.get("catalog")

spark.sql(f"USE CATALOG {catalog}")
print(f"Catalog: {catalog}")

In [ ]:
from pyspark.sql.functions import col

def transform(bronze_name, silver_name, transform_fn):
    df = spark.table(f"{catalog}.bronze.{bronze_name}")
    df_out = transform_fn(df).dropDuplicates()
    df_out.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.silver.{silver_name}")
    print(f"silver.{silver_name}: {df_out.count()} rijen")

In [ ]:
# ─── silver.persona ─────────────────────────────────────────────────
transform("persona_created", "persona", lambda df: df.select(
    col("bsn"), col("voornaam"), col("achternaam"), col("geslacht"),
    col("geboortedatum"), col("woonplaats"),
    col("source_ts").alias("ingestion_source_ts"), col("event_date"),
))

In [ ]:
transform("polisadm_ikv", "polisadm_ikv", lambda df: df.select(
    col("ikv_id"), col("bsn"), col("werkgever_naam"),
    col("aanvang_dienstverband"), col("einde_dienstverband"), col("loon_bruto_jaar"),
))
transform("ww_aanvraag", "ww_aanvraag", lambda df: df.select(
    col("aanvraag_id"), col("bsn"), col("aanvraag_datum"),
    col("status"), col("reden_einde_dienstverband"),
))
transform("zw_melding", "zw_melding", lambda df: df.select(
    col("melding_id"), col("bsn"), col("eerste_ziektedag"), col("duur_dagen"),
))
transform("wia_aanvraag", "wia_aanvraag", lambda df: df.select(
    col("aanvraag_id"), col("bsn"), col("aanvraag_datum"), col("status"),
    col("onderdeel"), col("arbeidsongeschikt_pct"), col("regio_code"),
))
transform("wajong_dossier", "wajong_dossier", lambda df: df.select(
    col("dossier_id"), col("bsn"), col("ingangsdatum"),
    col("regime"), col("arbeidsvermogen"),
))
transform("crm_contact", "crm_contact", lambda df: df.select(
    col("contact_id"), col("bsn"), col("contact_ts"),
    col("kanaal"), col("onderwerp"), col("duur_seconden"),
))
print("Silver klaar.")